# BEV_real L3 (auto): env -> nuScenes-mini -> L3C LiDAR-only -> L3D LiDAR+Camera
Pretrained BEVFusion inference + nuScenes eval. Heavy files live in /kaggle/temp (not saved as output); results go to /kaggle/working/outputs.

In [ ]:
%%bash
set -euo pipefail
cat > /kaggle/working/l3_env.sh <<'EOF'
export B=/kaggle/temp; [ -d "$B" ] || B=/tmp
export W=$B/l3
export ENVDIR=$W/envs/bevfusion
export PY=$ENVDIR/bin/python
export C=$W/constraints.txt
export CUDA_VISIBLE_DEVICES=0
export MPLBACKEND=Agg
export OUT=/kaggle/working/outputs
[ -d /usr/local/cuda ] && export CUDA_HOME=/usr/local/cuda
true
EOF
source /kaggle/working/l3_env.sh
mkdir -p $W/logs $W/exports $W/checkpoints $OUT
echo "W=$W"; df -h $W /kaggle/working | cat
nvidia-smi -L
which nvcc && nvcc --version | tail -2
git clone --depth 1 --branch v1.4.0 https://github.com/open-mmlab/mmdetection3d.git $W/mmdetection3d 2>&1 | tail -2
ls $W/mmdetection3d/projects/BEVFusion/configs

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/l3_env.sh
{
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O $W/miniconda.sh
bash $W/miniconda.sh -b -p $W/miniconda3 > /dev/null
$W/miniconda3/bin/conda --version
$W/miniconda3/bin/conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main || true
$W/miniconda3/bin/conda tos accept || true
$W/miniconda3/bin/conda create -y --override-channels -c conda-forge -p $ENVDIR python=3.10 pip
$PY --version
} 2>&1 | tee $W/logs/step_conda.log

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/l3_env.sh
cat > $C <<'EOF'
numpy==1.26.4
torch==2.1.0+cu121
torchvision==0.16.0+cu121
EOF
{
$PY -m pip install -q -U pip wheel
$PY -m pip install -c $C torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121
$PY -m pip install -c $C "numpy==1.26.4" ninja
$PY -m pip install -c $C "mmengine==0.10.3"
$PY -m pip install -c $C "mmcv==2.1.0" --only-binary=mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/index.html
$PY -m pip install -c $C spconv-cu120
$PY -m pip install "setuptools==69.5.1"
$PY -c "
import spconv, spconv.pytorch
print('spconv', spconv.__version__)
import torch, numpy, mmengine, mmcv, setuptools
from mmcv.ops import nms
assert numpy.__version__=='1.26.4' and torch.__version__=='2.1.0+cu121' and mmcv.__version__=='2.1.0' and setuptools.__version__=='69.5.1'
b = torch.tensor([[0,0,10,10],[1,1,11,11],[50,50,60,60]], dtype=torch.float32, device='cuda')
s = torch.tensor([0.9,0.8,0.7], device='cuda')
_, keep = nms(b, s, 0.5); assert keep.tolist()==[0,2], keep
print('ENV GATE PASS', torch.__version__, numpy.__version__, mmengine.__version__, mmcv.__version__, torch.cuda.get_device_name(0))
"
} 2>&1 | tee $W/logs/step_env.log

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/l3_env.sh
cd $W/mmdetection3d
{
$PY -m pip install -c $C "mmdet==3.3.0"
$PY -m pip install -c $C lyft_dataset_sdk "networkx>=2.5" numba nuscenes-devkit plyfile scikit-image tensorboard trimesh terminaltables
$PY -m pip install -c $C --no-deps --no-build-isolation -e .
$PY -W ignore -c "
import numpy, torch, setuptools, mmengine, mmcv, mmdet, mmdet3d, mmcv._ext
from mmdet3d.utils import register_all_modules
register_all_modules(init_default_scope=True)
from mmdet3d.datasets import NuScenesDataset
from mmdet3d.models.layers.spconv import IS_SPCONV2_AVAILABLE
print('IS_SPCONV2_AVAILABLE =', IS_SPCONV2_AVAILABLE)
assert IS_SPCONV2_AVAILABLE, 'mmdet3d khong dung spconv2 -> checkpoint se khong nap duoc'
v = dict(numpy=numpy.__version__, torch=torch.__version__, setuptools=setuptools.__version__, mmengine=mmengine.__version__, mmcv=mmcv.__version__, mmdet=mmdet.__version__, mmdet3d=mmdet3d.__version__)
print(v)
assert v['numpy']=='1.26.4' and v['torch']=='2.1.0+cu121' and v['mmcv']=='2.1.0' and v['mmdet']=='3.3.0' and v['mmdet3d']=='1.4.0', v
print('MMDET3D GATE PASS')
"
$PY -m pip check || true
} 2>&1 | tee $W/logs/step_mmdet3d.log

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/l3_env.sh
cd $W/mmdetection3d
LOG=$W/logs/step_ops.log
{
NVCC_MAJOR=$(nvcc --version | grep -oP 'release \K[0-9]+')
[ "$NVCC_MAJOR" = "12" ] || { echo "nvcc major=$NVCC_MAJOR != 12"; exit 1; }
export TORCH_CUDA_ARCH_LIST="7.5" FORCE_CUDA=1 MAX_JOBS=4
$PY -W ignore projects/BEVFusion/setup.py develop
} > $LOG 2>&1 || { grep -nE "error|Error|FAILED" $LOG | head -40; tail -40 $LOG; exit 1; }
tail -3 $LOG
$PY -W ignore -c "
import projects.BEVFusion.bevfusion
from mmdet3d.registry import MODELS
assert all(n in MODELS.module_dict for n in ['BEVFusion','DepthLSSTransform','TransFusionHead'])
print('BEVFUSION OPS GATE PASS')
" 2>&1 | tee -a $LOG

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/l3_env.sh
cd $W/mmdetection3d
export PYTHONPATH=$PWD:${PYTHONPATH:-}
V=$(find /kaggle/input -maxdepth 5 -type d -name v1.0-mini | head -1)
[ -n "$V" ] || { echo "khong thay v1.0-mini trong /kaggle/input"; find /kaggle/input -maxdepth 3 | head -30; exit 1; }
ROOT=$(dirname "$V"); echo "nuScenes-mini root: $ROOT"
DST=$PWD/data/nuscenes; mkdir -p $DST
for d in samples sweeps maps v1.0-mini; do ln -sfn "$ROOT/$d" "$DST/$d"; done
ls -la $DST
{
$PY -W ignore - <<'EOF'
import sys, os, inspect, time
sys.path.insert(0, os.getcwd())
sys.argv = ['create_data.py', 'nuscenes', '--root-path', './data/nuscenes', '--out-dir', './data/nuscenes',
            '--extra-tag', 'nuscenes', '--version', 'v1.0-mini']
from mmengine.registry import init_default_scope
init_default_scope('mmdet3d')
import tools.create_data as cd
sig = inspect.signature(cd.nuscenes_data_prep)
kw = dict(root_path='./data/nuscenes', info_prefix='nuscenes', version='v1.0-mini', out_dir='./data/nuscenes', max_sweeps=10)
if 'dataset_name' in sig.parameters: kw['dataset_name'] = 'NuScenesDataset'
kw = {k: v for k, v in kw.items() if k in sig.parameters}
print('call kwargs:', kw)
t = time.time(); cd.nuscenes_data_prep(**kw); print('done in', round(time.time() - t, 1), 's')
import mmengine
for s in ['train', 'val']:
    d = mmengine.load(f'data/nuscenes/nuscenes_infos_{s}.pkl')
    n = [len(x.get('lidar_sweeps', []) or []) for x in d['data_list']]
    print(s, len(d['data_list']), d['metainfo'], 'sweeps min/max', min(n), max(n))
EOF
test -f data/nuscenes/nuscenes_infos_val.pkl && echo "CREATE_DATA PASS"
} 2>&1 | tee $W/logs/step_data.log

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/l3_env.sh
cd $W/mmdetection3d
export PYTHONPATH=$PWD:${PYTHONPATH:-}
TAG=L3C_lidar_only
CFG=projects/BEVFusion/configs/bevfusion_lidar_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py
CKPT=$W/checkpoints/$(basename https://download.openmmlab.com/mmdetection3d/v1.1.0_models/bevfusion/bevfusion_lidar_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d-2628f933.pth)
[ -f "$CKPT" ] || wget -q https://download.openmmlab.com/mmdetection3d/v1.1.0_models/bevfusion/bevfusion_lidar_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d-2628f933.pth -O "$CKPT"
ls -la "$CKPT"
rm -rf $W/exports/$TAG $W/work_dirs/$TAG
$PY -W ignore tools/test.py $CFG $CKPT --work-dir $W/work_dirs/$TAG \
   --cfg-options test_evaluator.jsonfile_prefix=$W/exports/$TAG test_dataloader.dataset.metainfo.version=v1.0-mini 2>&1 | tee $W/logs/test_$TAG.log | grep -vE "^\s*$" | tail -60
mkdir -p $OUT/$TAG
if grep -q "size mismatch\|missing keys in source" $W/logs/test_$TAG.log; then echo "CHECKPOINT LOAD MISMATCH -> ket qua vo nghia"; grep -m5 "size mismatch\|missing keys" $W/logs/test_$TAG.log; fi
find $W/exports/$TAG -name "results_nusc.json" -exec cp {} $OUT/$TAG/ \;
find $W/exports/$TAG -name "metrics_summary.json" -exec cp {} $OUT/$TAG/ \;
cp $W/logs/test_$TAG.log $OUT/$TAG/
ls -la $OUT/$TAG
$PY - <<'EOF'
import json, glob
f = glob.glob('/kaggle/working/outputs/L3C_lidar_only/metrics_summary.json')
assert f, 'khong co metrics_summary.json -> inference/eval that bai'
m = json.load(open(f[0]))
print('L3C_lidar_only  mAP=%.4f  NDS=%.4f' % (m['mean_ap'], m['nd_score']))
log = open('/kaggle/working/outputs/L3C_lidar_only/test_L3C_lidar_only.log', errors='replace').read()
assert 'size mismatch' not in log, 'checkpoint size mismatch'
assert m['mean_ap'] > 0.2, 'mAP qua thap -> pipeline sai, khong phai PASS'
print('L3C_lidar_only INFERENCE PASS')
EOF

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/l3_env.sh
cd $W/mmdetection3d
export PYTHONPATH=$PWD:${PYTHONPATH:-}
TAG=L3D_lidar_camera
CFG=projects/BEVFusion/configs/bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d.py
CKPT=$W/checkpoints/$(basename https://download.openmmlab.com/mmdetection3d/v1.1.0_models/bevfusion/bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d-5239b1af.pth)
[ -f "$CKPT" ] || wget -q https://download.openmmlab.com/mmdetection3d/v1.1.0_models/bevfusion/bevfusion_lidar-cam_voxel0075_second_secfpn_8xb4-cyclic-20e_nus-3d-5239b1af.pth -O "$CKPT"
ls -la "$CKPT"
rm -rf $W/exports/$TAG $W/work_dirs/$TAG
$PY -W ignore tools/test.py $CFG $CKPT --work-dir $W/work_dirs/$TAG \
   --cfg-options test_evaluator.jsonfile_prefix=$W/exports/$TAG test_dataloader.dataset.metainfo.version=v1.0-mini 2>&1 | tee $W/logs/test_$TAG.log | grep -vE "^\s*$" | tail -60
mkdir -p $OUT/$TAG
if grep -q "size mismatch\|missing keys in source" $W/logs/test_$TAG.log; then echo "CHECKPOINT LOAD MISMATCH -> ket qua vo nghia"; grep -m5 "size mismatch\|missing keys" $W/logs/test_$TAG.log; fi
find $W/exports/$TAG -name "results_nusc.json" -exec cp {} $OUT/$TAG/ \;
find $W/exports/$TAG -name "metrics_summary.json" -exec cp {} $OUT/$TAG/ \;
cp $W/logs/test_$TAG.log $OUT/$TAG/
ls -la $OUT/$TAG
$PY - <<'EOF'
import json, glob
f = glob.glob('/kaggle/working/outputs/L3D_lidar_camera/metrics_summary.json')
assert f, 'khong co metrics_summary.json -> inference/eval that bai'
m = json.load(open(f[0]))
print('L3D_lidar_camera  mAP=%.4f  NDS=%.4f' % (m['mean_ap'], m['nd_score']))
log = open('/kaggle/working/outputs/L3D_lidar_camera/test_L3D_lidar_camera.log', errors='replace').read()
assert 'size mismatch' not in log, 'checkpoint size mismatch'
assert m['mean_ap'] > 0.2, 'mAP qua thap -> pipeline sai, khong phai PASS'
print('L3D_lidar_camera INFERENCE PASS')
EOF

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/l3_env.sh
$PY - <<'EOF'
import json
ref = {'L3C_lidar_only': (0.5431496439080831, 0.5625808309394464), 'L3D_lidar_camera': (0.5702214005656601, 0.5755269346010203)}
for k, (m0, n0) in ref.items():
    m = json.load(open(f'/kaggle/working/outputs/{k}/metrics_summary.json'))
    d1, d2 = abs(m['mean_ap'] - m0), abs(m['nd_score'] - n0)
    print(k, 'mAP', round(m['mean_ap'], 5), 'NDS', round(m['nd_score'], 5), '| delta', round(d1, 5), round(d2, 5))
    assert d1 < 5e-3 and d2 < 5e-3, 'REGRESSION: rebuilt env does not reproduce frozen L3C/L3D'
print('FREEZE REGRESSION PASS')
EOF

In [ ]:
%%bash
source /kaggle/working/l3_env.sh
{
echo "== date (UTC)"; date -u
echo "== host"; uname -a; head -3 /etc/os-release; python3 --version
echo "== GPU"; nvidia-smi; nvcc --version | tail -2
echo "== env python"; $PY --version
echo "== torch/cuda"; $PY -c "import torch,spconv;print('torch',torch.__version__,'cuda',torch.version.cuda,'cudnn',torch.backends.cudnn.version(),'spconv',spconv.__version__)"
echo "== mmdetection3d git"; git -C $W/mmdetection3d log -1 --format='%H %d' ; git -C $W/mmdetection3d describe --tags 2>/dev/null || true
echo "== constraints"; cat $C
echo "== env vars"; echo "MPLBACKEND=$MPLBACKEND CUDA_VISIBLE_DEVICES=$CUDA_VISIBLE_DEVICES CUDA_HOME=${CUDA_HOME:-}"
echo "== conda list --export"; $W/miniconda3/bin/conda list -p $ENVDIR --export
echo "== pip freeze (bevfusion env)"; $PY -m pip freeze
echo "== pip check"; $PY -m pip check || true
} > $OUT/environment.txt 2>&1
wc -l $OUT/environment.txt; grep -E "^(torch|numpy|mmcv|mmdet|mmdet3d|mmengine|spconv|setuptools)==|^mmdet3d" $OUT/environment.txt | head -20

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/l3_env.sh
cd $W
if command -v zstd >/dev/null 2>&1; then CMP="zstd -T0 -3"; EXT=zst; else CMP="gzip -1"; EXT=gz; fi
echo "compressor: $CMP"
tar -cf - envs/bevfusion mmdetection3d constraints.txt logs | $CMP > /kaggle/working/env_bevfusion.tar.$EXT
ls -la /kaggle/working/env_bevfusion.tar.$EXT
sha256sum /kaggle/working/env_bevfusion.tar.$EXT | tee $OUT/SHA256SUMS.txt
if [ "$EXT" = zst ]; then zstd -t /kaggle/working/env_bevfusion.tar.zst && echo "archive integrity OK"; else gzip -t /kaggle/working/env_bevfusion.tar.gz && echo "archive integrity OK"; fi
echo "FREEZE PACK PASS"